# Asistente Fiscal con Gemini, RAG y LangGraph

Agente experto para gestorías españolas. Asesora sobre obligaciones fiscales de **autónomos y sociedades**: cómo rellenar declaraciones, plazos del calendario fiscal y avisos de antelación.

**Stack:** Google Gemini · ChromaDB · LangGraph · LangChain

---

## Índice

1. [Instalación y configuración](#1-instalación-y-configuración)
2. [Carga y procesado de documentos](#2-carga-y-procesado-de-documentos)
3. [Creación de la base de conocimiento vectorial](#3-creación-de-la-base-de-conocimiento-vectorial)
4. [Diseño del agente LangGraph](#4-diseño-del-agente-langgraph)
5. [Lógica de avisos por antelación](#5-lógica-de-avisos-por-antelación)
6. [Demo interactiva](#6-demo-interactiva)

## 1. Instalación y configuración

In [ ]:
# %pip install langchain langchain-google-genai langchain-community langchain-text-splitters langgraph chromadb pypdf pdfplumber python-dotenv pandas sentence-transformers

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env")

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
assert GOOGLE_API_KEY, "Falta GOOGLE_API_KEY en el archivo .env"
print("API key cargada correctamente")

API key cargada correctamente


## 2. Carga y procesado de documentos

In [ ]:
from pathlib import Path
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
import pdfplumber

# Rutas organizadas por tipo
PRACTICOS_ES = Path("../data/manuales/practicos/es")
WEB_ES       = Path("../data/manuales/web/es")

# Metadatos por archivo: modelo fiscal cubierto, perfil, tipo e idioma
MANUAL_METADATA = {
    (PRACTICOS_ES, "manual_iva_303_2025.pdf"):                   {"modelos": "303",     "perfil": "ambos",    "tipo": "manual_practico", "idioma": "es"},
    (PRACTICOS_ES, "manual_actividades_economicas_111_115.pdf"):  {"modelos": "111,115", "perfil": "ambos",    "tipo": "manual_practico", "idioma": "es"},
    (PRACTICOS_ES, "manual_renta_100_130_2025_parte1.pdf"):       {"modelos": "100,130", "perfil": "autonomo", "tipo": "manual_practico", "idioma": "es"},
    (PRACTICOS_ES, "manual_renta_100_130_2025_parte2.pdf"):       {"modelos": "100,130", "perfil": "autonomo", "tipo": "manual_practico", "idioma": "es"},
    (PRACTICOS_ES, "manual_sociedades_200_202_2024.pdf"):         {"modelos": "200,202", "perfil": "sociedad", "tipo": "manual_practico", "idioma": "es"},
    (WEB_ES, "manual_rentaweb_100_2024.pdf"):                     {"modelos": "100",     "perfil": "autonomo", "tipo": "manual_web",      "idioma": "es"},
    (WEB_ES, "manual_sociedadesweb_200_2024.pdf"):                {"modelos": "200",     "perfil": "sociedad", "tipo": "manual_web",      "idioma": "es"},
}

CHROMA_DIR_CHECK = Path("../chroma_db")

# Si la base vectorial ya existe, saltamos la carga de PDFs por completo.
# La carga con pdfplumber puede tardar varios minutos y solo hace falta
# la primera vez que se crea el índice (o si se borra chroma_db/).
if CHROMA_DIR_CHECK.exists():
    docs_manuales = []
    print("Base de conocimiento ya indexada — carga de PDFs omitida.")
else:
    splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=100)

    def cargar_pdfs(metadata_map: dict) -> list:
        """Carga PDFs con pdfplumber, divide en chunks y añade metadatos."""
        docs = []
        for (directorio, filename), meta in metadata_map.items():
            path = directorio / filename
            if not path.exists():
                print(f"  [AVISO] No encontrado: {path}")
                continue
            texto_completo = []
            with pdfplumber.open(str(path)) as pdf:
                for page in pdf.pages:
                    texto = page.extract_text()
                    if texto:
                        texto_completo.append(texto)
            texto_unido = "\n\n".join(texto_completo)
            doc_base = Document(page_content=texto_unido, metadata={**meta, "fuente": filename})
            chunks = splitter.split_documents([doc_base])
            docs.extend(chunks)
            print(f"  [{meta['idioma']}] {filename}: {len(chunks)} chunks")
        return docs

    print("Cargando manuales...")
    docs_manuales = cargar_pdfs(MANUAL_METADATA)

    print(f"\nTotal chunks manuales: {len(docs_manuales)}")


In [ ]:
import pandas as pd
from langchain_core.documents import Document

CALENDARIO_PATH   = Path("../data/calendario_fiscal.csv")
OBLIGACIONES_PATH = Path("../data/obligaciones_perfil.csv")

if CHROMA_DIR_CHECK.exists():
    docs_calendario   = []
    docs_obligaciones = []
    print("CSVs omitidos — base de conocimiento ya indexada.")
else:
    def cargar_csv_como_docs(path: Path, tipo: str, sep: str = ",") -> list:
        df = pd.read_csv(path, sep=sep)
        docs = []
        for _, row in df.iterrows():
            contenido = " | ".join(f"{col}: {val}" for col, val in row.items() if pd.notna(val))
            meta = {"fuente": path.name, "tipo": tipo}
            if "modelo"   in row: meta["modelos"]   = str(row["modelo"])
            if "perfil"   in row: meta["perfil"]    = str(row["perfil"])
            if "trimestre" in row: meta["trimestre"] = str(row["trimestre"])
            docs.append(Document(page_content=contenido, metadata=meta))
        return docs

    docs_calendario   = cargar_csv_como_docs(CALENDARIO_PATH,   "calendario",         sep=",")
    docs_obligaciones = cargar_csv_como_docs(OBLIGACIONES_PATH, "obligaciones_perfil", sep=";")

    print(f"Chunks calendario:   {len(docs_calendario)}")
    print(f"Chunks obligaciones: {len(docs_obligaciones)}")
    print(f"\nTotal a indexar: {len(docs_manuales) + len(docs_calendario) + len(docs_obligaciones)}")


## 3. Creación de la base de conocimiento vectorial

In [ ]:
import chromadb
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

CHROMA_DIR = "../chroma_db"
COLLECTION_NAME = "base_fiscal"

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

if Path(CHROMA_DIR).exists():
    vectorstore = Chroma(
        persist_directory=CHROMA_DIR,
        embedding_function=embeddings,
        collection_name=COLLECTION_NAME
    )
    print(f"Base de conocimiento cargada desde disco: {vectorstore._collection.count()} documentos")
else:
    all_docs = docs_manuales + docs_calendario + docs_obligaciones
    print(f"Total documentos a indexar: {len(all_docs)}")
    print("Indexando en local... (puede tardar 5-10 min)")

    vectorstore = Chroma.from_documents(
        documents=all_docs,
        embedding=embeddings,
        persist_directory=CHROMA_DIR,
        collection_name=COLLECTION_NAME
    )
    print(f"\nBase de conocimiento creada: {vectorstore._collection.count()} documentos")


In [ ]:
# Verificar la colección con consultas de prueba
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

test_queries = [
    "¿Cuándo es el plazo del modelo 303 del primer trimestre?",
    "¿Cómo se calcula la base imponible del IVA?",
    "¿Qué obligaciones fiscales tiene un autónomo en el primer trimestre?",
]

for q in test_queries:
    print(f"\nConsulta: {q}")
    results = retriever.invoke(q)
    for r in results:
        print(f"  [{r.metadata.get('fuente', '?')}] {r.page_content[:120]}...")

# Diagnóstico: ver qué idioma tiene el texto extraído de los manuales de renta
print("\n\n--- DIAGNÓSTICO: primeras líneas del manual de renta ---")
if docs_manuales:
    for doc in docs_manuales[:3]:
        if "renta" in doc.metadata.get("fuente", ""):
            print(f"Fuente: {doc.metadata['fuente']}")
            print(f"Texto: {doc.page_content[:300]}")
            print("---")


Consulta: ¿Cuándo es el plazo del modelo 303 del primer trimestre?
  [manual_renta_100_130_2025_parte1.pdf] a. Periodo inicial: del 01-07-25 al 31-12-25 (184 días)
Es el periodo comprendido entre el día siguiente a la fecha de v...
  [manual_sociedades_200_202_2024.pdf] presentació de la declaració haurà de realitzar-se durant els vint primers dies naturals dels 
mesos d'abril, juliol, oc...
  [manual_renta_100_130_2025_parte1.pdf] b. Període final :  del 01-01-26 al 31-06-26 (181 dies)
És el període comprès entre el dia 1 de gener de 2026 i el dia d...
  [manual_renta_100_130_2025_parte1.pdf] 2.2 Interessos de demora corresponents a la deducció indeguda de 2023 
(50.000 euros) 
a. Període inicial : del 02-07-24...

Consulta: ¿Cómo se calcula la base imponible del IVA?
  [manual_actividades_economicas_111_115.pdf] 5.3.2 En qué consiste
Quien realice entregas de bienes o prestaciones de servicios repercutirá el tipo impositivo 
del I...
  [manual_actividades_economicas_111_115.pdf] 5.3

## 4. Diseño del agente LangGraph

In [ ]:
SYSTEM_PROMPT = """Eres un asesor fiscal experto de una gestoría española llamada GestorIA.
Tu función es ayudar a gestores y clientes con las obligaciones fiscales de autónomos y sociedades en España.

## ROL Y LÍMITES

Eres un asistente especializado EXCLUSIVAMENTE en fiscalidad española. No respondas preguntas fuera de este ámbito.
Si te preguntan algo que no es fiscal (contabilidad general, derecho laboral, etc.), indica amablemente que está fuera de tu alcance.

## IDIOMA

Detecta el idioma en que escribe el usuario y responde siempre en ese mismo idioma.
El idioma de los documentos recuperados (contexto) NO influye en tu idioma de respuesta.

## REGLA DE ORO: NO ALUCINACIONES

- Usa ÚNICAMENTE la información que aparece literalmente en el contexto proporcionado.
- Si un dato concreto (fecha, casilla, porcentaje, plazo) no aparece textualmente en el contexto, NO lo inventes ni lo inferías.
- En ese caso responde exactamente: "No dispongo de información suficiente sobre este punto en mi base de conocimiento. Te recomiendo consultar la sede electrónica de la AEAT (sede.agenciatributaria.gob.es) o al gestor responsable."

## IDENTIFICACIÓN DE PERFIL

- SIEMPRE identifica el perfil del cliente antes de responder: autónomo, sociedad, o ambos.
- Si el perfil NO está claro en la pregunta ni en el historial, PREGUNTA antes de dar cualquier información fiscal. No asumas.
- Una vez identificado el perfil, recuérdalo durante toda la conversación. No vuelvas a preguntarlo.

## ESTRUCTURA DE RESPUESTA

Organiza SIEMPRE tus respuestas en este orden:
1. **Perfil identificado** — una línea confirmando si es autónomo o sociedad.
2. **Obligaciones aplicables** — lista de modelos con descripción breve.
3. **Plazos** — fecha límite de presentación y, si aplica, fecha límite de domiciliación.
4. **Inicio de preparación recomendado** — fecha_limite menos dias_preparacion_recomendados del calendario.
5. **Fuente** — cita el documento o fichero del que proviene cada dato.

Si la pregunta no requiere todos los apartados (ej: solo pregunta por un plazo), omite los irrelevantes.

## PLAZOS Y ANTELACIÓN

- Cuando informes de un plazo, calcula y muestra siempre la fecha recomendada de inicio de preparación.
- Fórmula: fecha_inicio_preparacion = fecha_limite - dias_preparacion_recomendados
- Si el usuario pregunta "¿qué tengo pendiente este mes/trimestre?", lista TODAS las obligaciones próximas ordenadas por fecha límite.

## TONO

Profesional, claro y directo. Sin tecnicismos innecesarios. Usa listas y negritas para facilitar la lectura.
No uses frases de relleno como "¡Claro!", "¡Por supuesto!", "¡Espero haberte ayudado!".

---

## EJEMPLOS DE RESPUESTA CORRECTA

**Ejemplo 1 — Pregunta con perfil claro:**
Usuario: "Soy autónomo, ¿cuándo presento el 303 del 2T?"

Respuesta:
**Perfil:** Autónomo.
**Modelo 303 — Autoliquidación IVA 2T 2026:**
- Fecha límite: 20 de julio de 2026
- Domiciliación hasta: 15 de julio de 2026
- Inicio de preparación recomendado: 10 de julio de 2026 (10 días antes)
*Fuente: calendario_fiscal.csv*

---

**Ejemplo 2 — Perfil no especificado:**
Usuario: "¿Cuándo tengo que presentar el modelo 303?"

Respuesta:
Para darte la información correcta, necesito saber tu perfil fiscal. ¿Eres autónomo o representas a una sociedad?

---

**Ejemplo 3 — Información no disponible:**
Usuario: "¿Cuál es el tipo de recargo de equivalencia para el aceite de oliva?"

Respuesta:
No dispongo de información suficiente sobre este punto en mi base de conocimiento. Te recomiendo consultar la sede electrónica de la AEAT (sede.agenciatributaria.gob.es) o al gestor responsable.
"""

print("System prompt configurado.")


In [ ]:
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
import operator

class AgentState(TypedDict):
    messages: Annotated[list, operator.add]
    perfil: str          # "autonomo" | "sociedad" | "ambos" | ""
    contexto_rag: str

# gemini-2.5-flash es el modelo disponible en el free tier (20 req/día, 5 RPM)
# temperature=0: respuestas deterministas. En un agente fiscal no queremos variabilidad —
# cada fecha, casilla o porcentaje debe ser siempre el mismo independientemente del intento.
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=GOOGLE_API_KEY,
    temperature=0
)

def recuperar_contexto(state: AgentState) -> AgentState:
    """Recupera contexto de ChromaDB combinando dos búsquedas:
    1. Búsqueda semántica general en manuales (k=5), filtrada por perfil si está definido.
    2. Búsqueda forzada en calendario y obligaciones (k=6) para garantizar que los plazos
       siempre estén disponibles, ya que los chunks CSV compiten en similitud con los PDFs
       y quedarían fuera con un único retriever de k pequeño.
    """
    ultima_pregunta = state["messages"][-1].content
    perfil = state.get("perfil", "")

    # Búsqueda 1: manuales filtrados por perfil
    search_kwargs_manuales = {"k": 5}
    if perfil in ("autonomo", "sociedad"):
        search_kwargs_manuales["filter"] = {"perfil": {"$in": [perfil, "ambos"]}}
    retriever_manuales = vectorstore.as_retriever(search_kwargs=search_kwargs_manuales)
    docs_manuales_res = retriever_manuales.invoke(ultima_pregunta)

    # Búsqueda 2: siempre incluir chunks del calendario y obligaciones por perfil
    # Filtramos por tipo para asegurar que los plazos aparecen en el contexto
    filter_csv = {"tipo": {"$in": ["calendario", "obligaciones_perfil"]}}
    retriever_csv = vectorstore.as_retriever(search_kwargs={"k": 6, "filter": filter_csv})
    docs_csv_res = retriever_csv.invoke(ultima_pregunta)

    # Combinar resultados eliminando duplicados por contenido
    vistos = set()
    docs_combinados = []
    for doc in docs_manuales_res + docs_csv_res:
        clave = doc.page_content[:100]
        if clave not in vistos:
            vistos.add(clave)
            docs_combinados.append(doc)

    contexto = "\n\n".join(
        f"[{d.metadata.get('fuente', '?')}]\n{d.page_content}"
        for d in docs_combinados
    )
    return {"contexto_rag": contexto}

def generar_respuesta(state: AgentState) -> AgentState:
    """Genera la respuesta usando Gemini con el contexto RAG recuperado y el historial."""
    contexto = state.get("contexto_rag", "")
    historial = state["messages"]

    # Construir el prompt: system + historial previo + pregunta actual con contexto
    messages = [SystemMessage(content=SYSTEM_PROMPT)]
    messages += historial[:-1]

    ultima = historial[-1].content
    prompt_con_contexto = f"""Contexto recuperado de la base de conocimiento:
---
{contexto}
---

Pregunta del usuario: {ultima}"""

    messages.append(HumanMessage(content=prompt_con_contexto))
    respuesta = llm.invoke(messages)

    # Detectar el perfil del cliente automáticamente si no está definido
    perfil_actual = state.get("perfil", "")
    texto = ultima.lower()
    if not perfil_actual:
        if "autónomo" in texto or "autonomo" in texto:
            perfil_actual = "autonomo"
        elif "sociedad" in texto or "empresa" in texto or "s.l" in texto:
            perfil_actual = "sociedad"

    return {
        "messages": [AIMessage(content=respuesta.content)],
        "perfil": perfil_actual
    }

# Construir el grafo LangGraph: recuperar contexto → generar respuesta → fin
workflow = StateGraph(AgentState)
workflow.add_node("recuperar_contexto", recuperar_contexto)
workflow.add_node("generar_respuesta", generar_respuesta)

workflow.set_entry_point("recuperar_contexto")
workflow.add_edge("recuperar_contexto", "generar_respuesta")
workflow.add_edge("generar_respuesta", END)

# MemorySaver persiste el historial y el perfil del cliente entre turnos de conversación
memory = MemorySaver()
agente = workflow.compile(checkpointer=memory)

print("Agente LangGraph compilado correctamente con gemini-2.5-flash.")


## 5. Lógica de avisos por antelación

In [ ]:
from datetime import date, timedelta

def obtener_obligaciones_proximas(perfil: str, dias_horizonte: int = 60) -> str:
    """Devuelve las obligaciones fiscales próximas en los próximos N días."""
    df = pd.read_csv("../data/calendario_fiscal.csv", sep=",")
    hoy = date.today()
    limite = hoy + timedelta(days=dias_horizonte)

    if perfil in ("autonomo", "sociedad"):
        df = df[df["perfil"].isin([perfil, "ambos"])]

    df["fecha_limite_2026"] = pd.to_datetime(df["fecha_limite_2026"]).dt.date
    df = df[(df["fecha_limite_2026"] >= hoy) & (df["fecha_limite_2026"] <= limite)]
    df = df.sort_values("fecha_limite_2026")

    if df.empty:
        return f"No hay obligaciones fiscales en los próximos {dias_horizonte} días."

    lineas = [f"Obligaciones próximas ({hoy} → {limite}):\n"]
    for _, row in df.iterrows():
        fecha = row["fecha_limite_2026"]
        inicio = fecha - timedelta(days=int(row["dias_preparacion_recomendados"]))
        lineas.append(
            f"• Modelo {row['modelo']} — {row['nombre']}\n"
            f"  Plazo: {fecha} | Inicio recomendado: {inicio}\n"
        )
    return "\n".join(lineas)

# Ejemplo
print(obtener_obligaciones_proximas("autonomo", dias_horizonte=90))

Obligaciones próximas (2026-05-03 → 2026-08-01):

• Modelo 100 — IRPF anual 2025 — fin campaña
  Plazo: 2026-06-30 | Inicio recomendado: 2026-05-31

• Modelo 130 — Pago fraccionado IRPF autónomos 2T 2026
  Plazo: 2026-07-20 | Inicio recomendado: 2026-07-10

• Modelo 303 — Autoliquidación IVA 2T 2026
  Plazo: 2026-07-20 | Inicio recomendado: 2026-07-10

• Modelo 111 — Retenciones e ingresos a cuenta IRPF 2T 2026
  Plazo: 2026-07-20 | Inicio recomendado: 2026-07-15

• Modelo 115 — Retenciones e ingresos a cuenta — alquileres 2T 2026
  Plazo: 2026-07-20 | Inicio recomendado: 2026-07-15



## 6. Demo interactiva

Ejecuta la celda siguiente para abrir un chat con el agente. Escribe `salir` para terminar la sesión.

In [ ]:
import uuid

def chat(perfil_inicial: str = ""):
    """Sesión de chat interactiva con el agente fiscal."""
    thread_id = str(uuid.uuid4())
    config = {"configurable": {"thread_id": thread_id}}
    state = {"messages": [], "perfil": perfil_inicial, "contexto_rag": ""}

    print("=" * 60)
    print("  ASISTENTE FISCAL — Gestoría España")
    print("=" * 60)
    if perfil_inicial:
        print(f"  Perfil activo: {perfil_inicial.upper()}")
    print("  Escribe 'salir' para terminar.\n")

    while True:
        pregunta = input("Tú: ").strip()
        if pregunta.lower() in ("salir", "exit", "quit"):
            print("Sesión finalizada.")
            break
        if not pregunta:
            continue

        state["messages"] = state.get("messages", []) + [HumanMessage(content=pregunta)]
        result = agente.invoke(state, config=config)
        state = result

        respuesta = result["messages"][-1].content
        print(f"\nAsistente: {respuesta}\n")
        print("-" * 60)

chat(perfil_inicial="")

  ASISTENTE FISCAL — Gestoría España
  Escribe 'salir' para terminar.



---

### Casos de prueba documentados

Los 5 casos mínimos requeridos se prueban en las celdas siguientes (sin entrada interactiva).

In [ ]:
def preguntar(pregunta: str, state: dict, config: dict) -> tuple[str, dict]:
    state["messages"] = state.get("messages", []) + [HumanMessage(content=pregunta)]
    result = agente.invoke(state, config=config)
    return result["messages"][-1].content, result

# Sesión de prueba
thread_id = str(uuid.uuid4())
config = {"configurable": {"thread_id": thread_id}}
state = {"messages": [], "perfil": "", "contexto_rag": ""}

casos = [
    # Caso 1 — Plazo de un modelo concreto
    "¿Cuál es el plazo para presentar el modelo 303 del primer trimestre de 2026?",
    # Caso 2 — Cómo rellenar una casilla específica
    "¿Cómo se calcula la casilla 01 del modelo 303?",
    # Caso 3 — Obligaciones autónomo 1T
    "Soy autónomo en estimación directa. ¿Qué declaraciones tengo que presentar en el primer trimestre?",
    # Caso 4 — Obligaciones sociedad 2T
    "Somos una sociedad limitada. ¿Qué obligaciones fiscales tenemos en el segundo trimestre?",
    # Caso 5 — Pregunta encadenada (memoria)
    "¿Y cuándo debería empezar a preparar esas declaraciones para llegar a tiempo?",
]

for i, pregunta in enumerate(casos, 1):
    print(f"\n{'='*60}")
    print(f"CASO {i}: {pregunta}")
    print('='*60)
    respuesta, state = preguntar(pregunta, state, config)
    print(f"RESPUESTA:\n{respuesta}")